# Framework + R0 Baseline

This is my write-up of the shared repair framework and the R0 baseline for our DSA4213 project. I wanted it to be more than just a description, so I added runnable cells that actually exercise the code and show the output, rather than me just claiming it works.

All the cells below run **offline** — no paid API calls and no network needed — so anyone on the team (or the grader) can re-run them and get the same results. They cover:

1. Environment / data availability check.
2. English query loading from the bilingual Hugging Face dataset.
3. The unit-test suite (pass/fail counts, with the known environment errors explained).
4. The shared verifier bucketing genuine violations vs. technical/structural errors.
5. Personal constraints (`hard_logic_py`) evaluated directly.
6. An end-to-end R0 repair loop driven by a **mock LLM**.

I didn't re-run the **real GPT-4.1 end-to-end runs** in here, because they need our OpenRouter key and cost money every time, and a grader couldn't reproduce them anyway. Instead I kept the artifacts from the runs I did do (the per-iteration prompts, plans, errors, token usage and timing) under `cache/`, and I point to them where relevant.

> **Environment note.** These cells were run with Python 3.12.8 and both the English (`database_en`) and Chinese (`chinatravel/environment/database`) ChinaTravel sandboxes installed via `scripts/setup_project.py`. **English is the default experiment language** (`--lang en`): the run/eval scripts default to English and the proof cells below use English data, so the interface and all produced output are in English. With both sandboxes present the full test suite passes with no errors. (The earlier real GPT-4.1 runs under `cache/` were executed before English became the default and were in Chinese; they are kept as historical evidence of the repair loop, and a fresh English baseline would be produced for the final experiments.)

## Step 0: Environment & Data Check

First confirm the Python version, the pinned dependencies, which sandbox datasets are present, and whether the API keys are configured. The API key values are **never** printed — only whether the variable name exists in `.env`.

In [1]:
import os, platform

# Run from the repository root so `chinatravel` imports resolve.
if os.path.basename(os.getcwd()) != 'Agentic-Travel-Planner-Ablation-of-Verifications-' \
        and os.path.isdir('Agentic-Travel-Planner-Ablation-of-Verifications-'):
    os.chdir('Agentic-Travel-Planner-Ablation-of-Verifications-')

print('Python:', platform.python_version())
for mod in ['numpy', 'pandas', 'pyarrow', 'jsonschema']:
    try:
        m = __import__(mod)
        import importlib.metadata as md
        print(f'  {mod}:', md.version(mod))
    except Exception as e:
        print(f'  {mod}: NOT INSTALLED ({e})')

print('Chinese sandbox (database)    present:', os.path.isdir('chinatravel/environment/database'))
print('English sandbox (database_en) present:', os.path.isdir('chinatravel/environment/database_en'))

# Report API-key configuration by NAME only, never the value.
if os.path.exists('.env'):
    names = {l.split('=')[0].strip() for l in open('.env')
             if '=' in l and not l.strip().startswith('#')}
    for k in ['OPENROUTER_API_KEY', 'CHINATRAVEL_OPENAI_API_KEY']:
        print(f'  {k} configured in .env:', k in names)

Python: 3.12.8
  numpy: 1.26.4


  pandas: 2.2.3
  pyarrow: 19.0.1
  jsonschema: 4.23.0
Chinese sandbox (database)    present: True
English sandbox (database_en) present: True
  OPENROUTER_API_KEY configured in .env: True
  CHINATRAVEL_OPENAI_API_KEY configured in .env: True


### Proof: English queries load end-to-end

The ChinaTravel query files on Hugging Face are **bilingual** — each query carries a Chinese (`nature_language`) and an English (`nature_language_en`) version. With `--lang en`, the loader downloads the same dataset, uses the English query text, and translates the city names to English (via the canonical `CITY_NAMES` map) so they resolve against the English sandbox (`database_en`). This confirms an **English input** is what reaches the agent, so the generated itinerary is produced in English.


In [2]:
import io, contextlib
from types import SimpleNamespace
from chinatravel.data.load_datasets import load_query

# Load the 'medium' split in English straight from the bilingual HF dataset.
args = SimpleNamespace(splits='medium', lang='en', oracle_translation=True)
with contextlib.redirect_stdout(io.StringIO()):
    ids, data = load_query(args)

print(f'Loaded {len(ids)} English queries from the "medium" split.')
q = data['e20241028160845920703']
print('Example query (English):')
print('  start_city :', q['start_city'])
print('  target_city:', q['target_city'])
print('  request    :', q['nature_language'][:150])
print('  constraints:', q['hard_logic_py'][:2], '...')

Loaded 150 English queries from the "medium" split.
Example query (English):
  start_city : Shenzhen
  target_city: Guangzhou
  request    : Current location: Shenzhen. I plan to visit Guangzhou for two days with a budget of 1000 RMB. I want to eat hotpot and book a single-bed room. Please 
  constraints: ['result=(day_count(plan)==2)', 'result=(people_count(plan)==1)'] ...


## Step 1: Understanding the original repair loop

Before changing anything, I spent time reading how the original ChinaTravel agent repairs a plan. The relevant file is `chinatravel/agent/nesy_verifier/llm_modulo.py`, and the main method is `LLMModuloAgent.solve(problem, prob_idx, oracle_verifier)`.

What I found is that the original R0 doesn't actually have a dedicated repair function. It "repairs" a plan by just prompting the LLM again, this time giving it the previous plan plus the verifier's feedback.

**Flow:**

1. **Collect information:** Shortlist transport, hotels, attractions and restaurants from the ChinaTravel environment.
2. **Generate:** Use `INITIAL_PROMPT` to generate the initial itinerary (Step 0).
3. **Verify:** Run `collect_commonsense_constraints_error` and `collect_personal_error` to identify violations.
4. **Repair:** If errors are reported, use `BACK_PROMPT` to generate a new itinerary using the previous plan and feedback.
5. **Repeat:** Continue until no errors are reported or the repair limit is reached.
6. **Return:** Output the final itinerary and verification results.

**Key points:**

- `max_steps` is the maximum number of repair attempts, controlled by `--refine_steps` (default 10).
- R0 regenerates the entire itinerary during repair. It does not protect previously satisfied constraints.
- The original verifier returns `error_info`, a list of reported violations. An empty list means no errors were reported, but some checks may have been skipped.
- Returning `True` from `solve()` does not necessarily mean all constraints passed.

## Step 2: Setting Up and Testing R0

### 2.1 Environment Setup

Set up ChinaTravel using Python 3.12.8 and the dependencies in `requirements.txt`.

**Configuration:**

- **Dataset:** ChinaTravel dataset from Hugging Face
- **Sandbox:** ChinaTravel-Sandbox. Both the English (`database_en`) and Chinese (`database`) sandboxes are installed; **English is the default** language for our experiments (`--lang en`).
- **LLM:** GPT-4.1 through OpenRouter
- **Baseline:** LLM-Modulo (R0)

**Issues encountered and fixes:**

1. **Dependencies:** Used NumPy 1.26.4 and PyArrow 19.0.1. Fixed the missing `_lzma` module by reinstalling Python with XZ support.
2. **Dataset format:** `hard_logic_py` was stored as a string instead of a list. Updated `load_datasets.py` to convert it using `ast.literal_eval()`.
3. **Missing sandbox:** Downloaded the official ChinaTravel-Sandbox. `scripts/setup_project.py` installs both the English (`database_en`) and Chinese (`database`) sandboxes.
4. **Missing import:** Added `import sys` to `llm_modulo.py`.
5. **OpenRouter:** Updated the model configuration to support `openai/gpt-4.1`.
6. **English as default:** The run/eval scripts (`run_exp.py`, `run_tpc.py`, `eval_exp.py`, etc.) now default to `--lang en`, so the interface and all produced output are in English unless `--lang zh` is passed explicitly.

I also added `scripts/setup_project.py` to automate sandbox installation and `.env.example` for API configuration. Teammates can follow the setup instructions in `README.md` instead of repeating these steps manually.

### 2.2 Testing the Original R0 (real GPT-4.1 runs)

Ran the original R0 using `--refine_steps 2`. These used the real API, so the proof is the saved run artifacts under `cache/` rather than live output.

**Test 1 — Easy split.** Case `e20241028160248698752`: one-day Shanghai→Hangzhou trip, budget 1,500 RMB. The initial itinerary passed the reported checks; no repair was needed (`ITER 0 ERROR: []`).

**Test 2 — Medium split.** Case `e20241028160845920703`: two-day Shenzhen→Guangzhou trip, budget 1,000 RMB, hotpot preference and single-bed room. The initial itinerary violated a dinner-timing constraint; R0 regenerated the itinerary and after one repair the verifier returned `ITER 1 ERROR: []`. Personal constraints were not fully checked here because `hard_logic_nl` was missing — one of the motivations for the shared verifier below.

The cell below lists the saved artifacts from these runs as evidence.

In [3]:
import os, glob

# Saved artifacts from the real GPT-4.1 R0 runs (prompts, plans, errors, logs).
roots = glob.glob('cache/*oracletranslation*/llm_modulo-openai/gpt-4.1')
if not roots:
    print('No cached GPT-4.1 run artifacts found under cache/.')
for root in roots:
    print('Run cache:', root)
    files = sorted(os.listdir(root))
    for f in files[:12]:
        print('   ', f)
    if len(files) > 12:
        print(f'    ... and {len(files) - 12} more files')

Run cache: cache/LLM-modulo_openai_gpt-4.1_2steps_oracletranslation_R1/llm_modulo-openai/gpt-4.1
Run cache: cache/LLM-modulo_openai_gpt-4.1_2steps_oracletranslation_R0/llm_modulo-openai/gpt-4.1
    problem_e20241028160845920703.error
    problem_e20241028160845920703.log
    problem_e20241028160845920703_repair_history.json
    problem_e20241028160845920703_step_0_prompt.txt
Run cache: cache/LLM-modulo_openai_gpt-4.1_en_5steps_oracletranslation_R0/llm_modulo-openai/gpt-4.1
    problem_e20241028160848776495.error
    problem_e20241028160848776495.log
    problem_e20241028160848776495_repair_history.json
    problem_e20241028160848776495_step_0_prompt.txt
    problem_e20241028160851829629.error
    problem_e20241028160851829629.log
    problem_e20241028160851829629_repair_history.json
    problem_e20241028160851829629_step_0_prompt.txt
    problem_e20241028160857444675.error
    problem_e20241028160857444675.log
    problem_e20241028160857444675_repair_history.json
    problem_e202410281

### 2.3 Logging

The original ChinaTravel implementation saves files for each iteration:

| File | Purpose |
|---|---|
| `step_0_prompt.txt` | Initial generation prompt |
| `step_0_plan.json` | Initial itinerary and verification results |
| `step_0_error.err` | Initial verifier feedback |
| `step_1_...` | Corresponding files after repair |

The saved plans also contain token usage and execution time. The original logs do not directly track constraint transitions. We added this to `shared_repair_loop.py`, which records which personal constraints were fixed, newly violated, still failed or still satisfied (demonstrated in Step 3).

## Step 3: Building the shared framework

The main thing I built was pulling the repair process out of the original agent, so that R0–R4 can all share the same planner, verifier and repair loop. The idea is that my teammates only have to write their strategy's `repair()` method and not reinvent the rest.

### 3.1 Files Added

All in `chinatravel/agent/nesy_verifier/`:

| File | Purpose |
|---|---|
| `shared_verifier.py` | Checks commonsense and personal constraints |
| `shared_repair_loop.py` | Handles repair attempts, verification, acceptance/rejection and logging |
| `repair_strategies/base.py` | Defines the common interface for R0–R4 |
| `repair_strategies/r0.py` | Implements R0 whole-plan regeneration |

We also updated `run_exp.py`, `load_model.py` and `llm_modulo.py` to support selecting repair strategies.

### 3.2 How the Shared Framework Works

1. ChinaTravel generates the initial itinerary.
2. The shared verifier checks commonsense and personal constraints.
3. The selected repair strategy proposes a revised itinerary.
4. The shared verifier checks the proposed repair.
5. The strategy decides whether to accept or reject it.
6. Repeat until verification passes or the repair limit is reached.

The original ChinaTravel R0 loop remains available when no repair strategy is selected.

### 3.3 Improvements Added

- **Shared verification:** Uses `hard_logic_py` directly to check personal constraints, even when `hard_logic_nl` is missing.
- **Raw itinerary preservation:** Keeps the original itinerary separate from the translated transport version.
- **Acceptance/rejection:** Supports accepting or rejecting repair proposals and keeping the previous plan when a proposal is rejected. The decision is left to each strategy.
- **Constraint-transition logging:** Records which personal constraints were fixed, newly violated, still failed, still passed or could not be evaluated.
- **Verifier reliability:** Separates genuine constraint violations from technical errors (verifier exceptions, transport translation failures, invalid itinerary structures).
- **Separate outputs:** Different repair strategies have separate result and log directories.

The shared framework does not automatically repair invalid plans, protect constraints or reject harmful changes. Those decisions belong to the individual R0–R4 strategies.

### 3.4 Proof: Unit-test suite

Run the full test suite for the shared framework and evaluation code. The cell reports the real counts and prints any errors.

**Expected:** with both the Chinese and English sandboxes installed (via `scripts/setup_project.py`), the full suite passes with no failures and no errors. If the English sandbox (`database_en`) is missing, three data-dependent tests would surface as *errors* (missing data file) rather than genuine *failures*; the cell distinguishes the two so a missing-data situation is never mistaken for a code defect.

In [4]:
import unittest, io, contextlib, os

# The test suite prints its own progress/logging to stdout; redirect it so
# only the clean summary below is shown. The result object is unaffected.
loader = unittest.TestLoader()
suite = loader.discover('tests', pattern='test_*.py')
buf = io.StringIO()
with open(os.devnull, 'w') as devnull, contextlib.redirect_stdout(devnull), contextlib.redirect_stderr(devnull):
    result = unittest.TextTestRunner(stream=buf, verbosity=0).run(suite)

print(f'Tests run : {result.testsRun}')
print(f'Failures  : {len(result.failures)}')
print(f'Errors    : {len(result.errors)}')
print(f'Passed    : {result.testsRun - len(result.failures) - len(result.errors)}')
print()
if result.errors:
    print('Errors (with both sandboxes installed, expect none; any here are')
    print('typically missing-data / environment issues, not code defects):')
    for test, _ in result.errors:
        print('   -', test.id())
if result.failures:
    print('Genuine failures:')
    for test, _ in result.failures:
        print('   -', test.id())
else:
    print('No genuine test failures.')

Tests run : 193
Failures  : 0
Errors    : 0
Passed    : 193

No genuine test failures.


### Proof: Shared verifier buckets errors correctly

We feed the verifier a deliberately **broken** itinerary: a standalone `innercity_transport` activity (which should be nested inside a destination activity's `transports` field) and an activity missing `start_time`. The verifier should:

- flag the structural problems in `schema_errors`,
- keep the "no translation function provided" notice in `translation_errors`,
- report genuine commonsense violations in `commonsense_errors`,
- and mark the plan `passed = False`.

This shows technical/structural problems are reported on their own channels and never silently pass.

In [5]:
from chinatravel.agent.nesy_verifier.shared_verifier import verify_plan

# English data: Shanghai -> Suzhou, with 'Humble Administrator's Garden'
# (the English sandbox name). Structural checks are language-independent; the
# commonsense verifier resolves the English city/POI against database_en.
problem = {'people_number': 2, 'start_city': 'Shanghai', 'target_city': 'Suzhou', 'hard_logic_py': []}

broken_plan = {
    'people_number': 2, 'start_city': 'Shanghai', 'target_city': 'Suzhou',
    'itinerary': [
        {'day': 1, 'activities': [
            # WRONG: inner-city transport as a standalone activity.
            {'type': 'innercity_transport', 'start_time': '08:00', 'end_time': '08:20'},
            # WRONG: attraction missing start_time.
            {'type': 'attraction', 'position': "Humble Administrator's Garden", 'end_time': '11:00'},
        ]},
    ],
}

import io, contextlib
# Suppress the verifier's incidental stdout so only the result buckets show.
with contextlib.redirect_stdout(io.StringIO()):
    res = verify_plan(problem, broken_plan, translate_fn=None)
print('passed            :', res['passed'])
print('schema_errors     :', len(res['schema_errors']))
for e in res['schema_errors']:
    print('    -', e)
print('translation_errors:', res['translation_errors'])
print('commonsense_errors:', len(res['commonsense_errors']))
print('verifier_errors   :', len(res['verifier_errors']))

passed            : False
schema_errors     : 2
    - Day 1 activity 1 has unsupported type 'innercity_transport'; inner-city transport must be nested in a destination activity's 'transports' field, not a separate activity.
    - Day 1 activity 2 ('attraction') is missing 'start_time'.
translation_errors: ['No transport translation function provided.']
commonsense_errors: 4
verifier_errors   : 1


### Proof: Personal constraints (`hard_logic_py`) evaluated directly

The shared verifier evaluates each `hard_logic_py` expression against the plan and returns `True` (pass), `False` (fail) or `None` (could not be evaluated). Here we pass one constraint that should hold and one that should not.

In [6]:
from chinatravel.agent.nesy_verifier.shared_verifier import check_personal_constraints

good_plan = {
    'people_number': 2, 'start_city': 'Shanghai', 'target_city': 'Suzhou',
    'itinerary': [
        {'day': 1, 'activities': [
            {'type': 'attraction', 'position': "Humble Administrator's Garden",
             'start_time': '09:00', 'end_time': '11:00', 'transports': 'empty'},
        ]},
    ],
}
prob_pc = {
    'people_number': 2, 'start_city': 'Shanghai', 'target_city': 'Suzhou',
    'hard_logic_py': [
        "result=(plan['people_number']==2)",   # expect True
        "result=(plan['people_number']==5)",   # expect False
    ],
}
results, eval_errors = check_personal_constraints(prob_pc, good_plan)
print('per-constraint results (True=pass, False=fail, None=uneval):', results)
print('evaluation_errors:', eval_errors)

per-constraint results (True=pass, False=fail, None=uneval): [True, False]
evaluation_errors: []


### Proof: End-to-end R0 repair loop (mock LLM, no API cost)

We run the real `run_repair_loop` with the real `R0WholePlanRepair` strategy, but drive it with a **mock LLM** so no network/API call is made. This proves the loop wiring end-to-end: it verifies the initial plan, calls the strategy once per repair step, re-verifies, applies the accept/reject decision, and records a step-by-step history.

The history table below is the **constraint-transition logging** in action: each step records the accept decision, the `passed` flag, and the per-constraint results.

In [7]:
from chinatravel.agent.nesy_verifier.repair_strategies.r0 import R0WholePlanRepair
from chinatravel.agent.nesy_verifier.shared_repair_loop import run_repair_loop

start_itinerary = [
    {'day': 1, 'activities': [
        {'type': 'attraction', 'position': "Humble Administrator's Garden",
         'start_time': '09:00', 'end_time': '11:00', 'transports': 'empty'},
    ]},
]
prob_e2e = {
    'people_number': 2, 'start_city': 'Shanghai', 'target_city': 'Suzhou',
    'nature_language': 'demo query',
    'hard_logic_py': ["result=(len(plan['itinerary'])>=1)"],
}
context = {k: '' for k in [
    'go_train_info', 'go_flight_info', 'back_train_info', 'back_flight_info',
    'attraction_info', 'restaurant_info', 'accommodation_info', 'innercity_transport_info',
]}

class MockLLM:
    calls = 0
    def __call__(self, messages, one_line=False):
        MockLLM.calls += 1
        # R0 prepends '[{"day": 1,' so we return the rest of a valid 1-day plan.
        return (' "activities": [{"type": "attraction", "position": "Humble Administrator\'s Garden",'
                ' "start_time": "09:00", "end_time": "11:00", "transports": "empty"}]}]')

def identity_translate(itinerary, problem):
    return itinerary  # offline stand-in for transport translation

import io, contextlib
# Suppress the verifier's incidental stdout; the history table below is the proof.
with contextlib.redirect_stdout(io.StringIO()):
    final = run_repair_loop(
        problem=prob_e2e, raw_plan=start_itinerary, strategy=R0WholePlanRepair(),
        context=context, llm=MockLLM(), translate_fn=identity_translate, max_steps=2,
    )

print('mock LLM calls         :', MockLLM.calls)
print('history steps recorded :', len(final['history']))
print()
print(f"{'step':>4} | {'accepted':>8} | {'passed':>6} | personal | schema_errors")
print('-' * 54)
for r in final['history']:
    v = r.get('verification', {})
    print(f"{r.get('step'):>4} | {str(r.get('accepted')):>8} | "
          f"{str(v.get('passed')):>6} | {str(v.get('personal_constraints')):>8} | "
          f"{len(v.get('schema_errors', []))}")

mock LLM calls         : 2
history steps recorded : 3

step | accepted | passed | personal | schema_errors
------------------------------------------------------
   0 |     True |  False |   [True] | 0
   1 |     True |  False |   [True] | 0
   2 |     True |  False |   [True] | 0


### 3.5 Making the verifier more reliable

Once R0 was running, I noticed three things that could quietly mess up our experiments if we didn't catch them. I fixed each one in `shared_verifier.py`, and I was careful **not** to touch the original ChinaTravel verifier or the repair strategies themselves — I only added detection on top.

1. **Commonsense verifier exceptions.** The original verifier sometimes returned Python exceptions as normal error messages (e.g. `KeyError: 'position'`). `_split_commonsense_errors()` separates these into `verifier_errors`, leaving genuine violations in `commonsense_errors`.
2. **Silent transport translation failures.** ChinaTravel's translation can silently replace a failed translation with an empty list. `_detect_silent_translation_failures()` compares transport info before and after translation and records failures in `translation_errors`.
3. **Invalid itinerary structures.** The LLM sometimes emitted standalone `innercity_transport` activities instead of nesting transport inside the destination activity. `_validate_plan_structure()` detects unsupported activity types, missing required fields and bad structure, recorded in `schema_errors`.

**Important:** these checks only *detect and report* problems. They do not auto-fix itineraries or change ChinaTravel's original constraint rules. (The error-bucketing cell above exercises checks 2 and 3 directly.)

### 3.6 Testing After Reliability Improvements

We added `tests/test_shared_verifier_reliability.py` to cover the new checks. The unit-test cell in 3.4 runs these alongside the rest of the suite.

The real GPT-4.1 end-to-end rerun of the medium case `e20241028160845920703` (two repair attempts) confirmed that: genuine constraint violations stayed detectable; technical/structural errors were recorded on their own channels; the final itinerary was still marked invalid when constraints were violated; R0 kept accepting proposals without protecting previously satisfied constraints; and constraint-transition logging correctly identified newly introduced violations (for example, the accommodation room-count constraint passed initially but failed after the first repair and stayed violated). This illustrates how whole-plan regeneration can introduce new violations while fixing others — a key motivation for R1–R4.

**Current status:** the shared framework and R0 pass the offline integration and reliability tests shown above. The framework is ready for R1–R4 development; the final comparative experiments across strategies are future work.

### 3.7 A bug I hit when running R0 for real (and the actual experiment)

When I first tried to run R0 live on real queries, every itinerary came back **empty**. It took
some digging to find the cause, and it wasn't the agent logic at all — it was a library version
clash. The newer `openai` package uses its own bundled copy of `httpx` (imported as `httpx2`),
but our usage-instrumentation transport in `chinatravel/agent/model_usage.py` was built against
the old `httpx`. Mixing the two made every model request fail silently, so the agent just
produced empty plans.

I fixed `model_usage.py` so the instrumented transport is built against whichever `httpx` the
installed `openai` client actually uses (while keeping the old path for the offline mock tests).
After that, live GPT-4.1 calls work again and the full offline suite stays green (193 tests).
It's a good reminder that 'empty output' can be a plumbing problem, not a modelling one.

**The actual R0 experiment** (running R0 on the train split and saving results) is documented in a
separate notebook, `train_test_split_r0.ipynb`. That notebook builds a deterministic train/test
split, runs R0 on the `medium_train` queries one by one, and commits the results to
`experiments/r0_medium_train/` so teammates get identical files without paying for the LLM again.
It also explains an important, initially-confusing result: R0 scores a **0% full-pass rate** on the
sample, because it lets the model write the transport details between places and the model guesses
some of them wrong (e.g. the wrong metro fare), which the verifier then rejects. That's the
expected weakness of the naive baseline — exactly the gap R1–R4 are meant to close.

## Step 4: How to get started (for my teammates)

This part is mostly notes for the others on the team who are implementing R1–R4.

### 4.1 Setting up

The framework is on the `main` branch. Follow the **Quick Setup** in `README.md`: clone and install dependencies, download the sandbox with `scripts/setup_project.py`, put your own OpenRouter key in `.env` (see `.env.example`), and run one R0 case to check your environment is working.

### 4.2 Main Files to Look At

| File | What to look at |
|---|---|
| `repair_strategies/base.py` | Shared repair interface |
| `repair_strategies/r0.py` | Example repair implementation |
| `shared_repair_loop.py` | How repairs are verified, accepted and logged |
| `shared_verifier.py` | Available constraint feedback |

You do not need to rewrite the original ChinaTravel planner.

### 4.3 Implementing Your Repair Strategy

Create your assigned strategy file inside `repair_strategies/` (`r1.py`, `r2.py`, `r3.py` or `r4.py`). Each strategy inherits from `BaseRepairStrategy` and implements `repair`:

```python
def repair(self, problem, raw_plan, verification_results, context, llm):
    # Your repair logic here.
    return RepairResult(repaired_plan=new_plan, metadata={"strategy": "R1"})
```

**Inputs:** `problem` (request + constraints), `raw_plan` (itinerary before transport translation), `verification_results` (shared-verifier feedback), `context` (shortlisted travel info), `llm` (GPT-4.1 adapter). **Output:** a `RepairResult` with the proposed raw itinerary and metadata. Strategies can override `accept()` to reject certain proposals.

### 4.4 Strategy Responsibilities

| Strategy | Main task |
|---|---|
| R1 | Repair only affected itinerary slots |
| R2 | Protect previously satisfied constraints and reject harmful repairs |
| R3 | Start with targeted repair and expand the scope when needed |
| R4 | Use the constraint-interdependency matrix to guide repair |

### 4.5 Running Experiments

After your strategy is implemented and registered in `llm_modulo.py`, run it with `--repair_strategy`:

```bash
python run_exp.py \
  --agent LLM-modulo \
  --repair_strategy R0 \
  --llm openai/gpt-4.1 \
  --splits medium \
  --index e20241028160845920703 \
  --refine_steps 5 \
  --oracle_translation
```

`--lang en` is now the default, so it no longer needs to be passed (add `--lang zh` only if you specifically want the Chinese sandbox). For the comparative experiments, keep the repair budget fixed at `--refine_steps 5` across every strategy (R0–R4) so comparisons are fair and R3 has room to escalate its repair scope.

Currently only the new R0 is connected. R1–R4 still need to be implemented, registered and tested.

### 4.6 Things to Take Note Of

- Use the shared verifier and repair loop rather than creating a separate agent.
- Keep the itinerary in ChinaTravel's expected format. Local transport belongs inside the `transports` field of an activity.
- Use the same model, dataset and repair budget for fair comparisons.
- Rejected proposals still count toward the repair budget.
- Track both fixed constraints and newly introduced violations.
- Test your strategy with a mock LLM (as in Step 3) before making paid API calls.
- The commonsense verifier can still produce execution errors and transport translation may fail silently; these need further validation before final experiments.

**For now, focus on implementing and testing your assigned repair strategy.** We can integrate and evaluate all methods together once R1–R4 are ready.